<a href="https://colab.research.google.com/github/kimjiwoo2/Pill-agent/blob/develop/notebooks/yoonsoo/ys_pretrained_ocr_score_testset_stage_ablation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pretrained OCR — Test Set 전처리 단계별 ablation (개선 지표 표 앞 3행 채우기용)

**목적**: "개선 지표" 표의 앞 3행(아무것도 없을 때 / CLAHE+unsharp / CLAHE+unsharp+upscale)을
**held-out 테스트셋(`Pillot/dataset/pilliot_test_set`, 5,088건)** 기준으로 계산한다. 표의 나머지
두 행(순수 multi-angle 0.787/0.617, polygon 기반 회전 보정+flip 0.834/0.720)은 이미 계산돼 있으므로
이 노트북에서는 다루지 않는다.

**3개 stage (누적)**:
1. **아무것도 없을 때**: 원본 이미지를 어떤 보정도 없이 그대로 OCR에 입력
2. **+ CLAHE+unsharp**: 명암 대비(clip=2.5, tile=8×8) + 선명화(sigma=1.5, strength=1.0)만 추가
3. **+ upscale(및 방향 정렬)**: `align_to_long_axis`(세로로 긴 이미지 90도 정렬) + `upscale_if_small`
   (면적 ≤71,818px² 시 2배 확대)까지 추가 — 이게 실제 파이프라인에서 multi-angle 탐색 직전에
   거치는 `load_and_prepare`와 동일한 상태라, 다음 stage(순수 multi-angle, 이미 계산됨)로 자연스럽게
   이어진다.

**회전 탐색(multi-angle) 없음**: 이 3개 stage는 전부 단일 추론(예측 1회)만 수행한다 — 회전 관련
개선은 이미 별도로 측정됐으므로 여기서는 순수 전처리 효과만 격리해서 본다. 그래서 후보 3개 ×
예측 1회로 가벼워서 전체 테스트셋(5,088건)을 샘플링 없이 그대로 돌릴 수 있다.


## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

Mounted at /content/drive


In [ ]:
# 파인튜닝 노트북과 동일 조치: paddleocr 내부에서 torch를 건드리는 경로가 있는데,
# Colab 기본 torch와 CUDA/NCCL 버전이 안 맞으면 'undefined symbol: ncclCommShrink' 류의
# ImportError가 남.
!pip uninstall -y torch torchvision torchaudio modelscope
!pip install paddlepaddle-gpu==3.1.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu118/
!pip install 'paddleocr==3.7.0'  # 버전 고정: 파인튜닝 노트북과 동일
!pip install -q opencv-python-headless pandas numpy matplotlib tqdm

Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cu118/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 GB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 15.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 15.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 699.9/699.9 MB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# install 셀 이후에 import해야 방금 설치한 버전이 로드됨
import re
import numpy as np
import pandas as pd
import cv2
import zipfile
from tqdm.auto import tqdm
from IPython.display import display

## 1. 경로 설정 (held-out 테스트셋)

In [ ]:
DRIVE_ROOT   = Path('/content/drive/MyDrive')
TEST_ZIP     = DRIVE_ROOT / 'Pillot' / 'dataset' / 'pilliot_test_set' / 'test_filtered.zip'
TEST_MANIFEST_CSV = DRIVE_ROOT / 'Pillot' / 'dataset' / 'pilliot_test_set' / 'final_test_manifest.csv'
CROP_DIR     = Path('/content/test_filtered')
RESULT_DIR   = DRIVE_ROOT / 'Pillot' / 'yoonsoo' / 'results'
RESULT_DIR.mkdir(parents=True, exist_ok=True)

if not CROP_DIR.exists() or not any(CROP_DIR.iterdir()):
    print('test_filtered 압축 해제 중...')
    with zipfile.ZipFile(TEST_ZIP, 'r') as zf:
        zf.extractall(CROP_DIR)

crop_paths = sorted(CROP_DIR.glob('*.png'))
print(f'TEST_ZIP: {TEST_ZIP}  exists={TEST_ZIP.exists()}')
print(f'테스트 이미지 {len(crop_paths)}개 발견')

# split으로 나누지 않고 전체 사용 (final_test_manifest.csv는 통째로 테스트셋)
_manifest_test = pd.read_csv(TEST_MANIFEST_CSV, low_memory=False)
print(f'final_test_manifest.csv 로드: {len(_manifest_test)}행')

test_filtered 압축 해제 중...
TEST_ZIP: /content/drive/MyDrive/Pillot/dataset/pilliot_test_set/test_filtered.zip  exists=True
테스트 이미지 5088개 발견
final_test_manifest.csv 로드: 5088행


## 2. 전처리 함수 (3개 stage, 누적)

In [ ]:
def load_raw(img_path):
    """stage 1: 원본 그대로 — 포맷 통일(BGR 3채널)만 하고 어떤 보정도 하지 않음."""
    img = cv2.imread(str(img_path), cv2.IMREAD_UNCHANGED)
    if img is None:
        raise FileNotFoundError(img_path)
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    elif img.shape[2] == 4:
        img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)
    return img


def apply_clahe_unsharp(image_bgr, clahe_clip=2.5, clahe_tile=8, unsharp_strength=1.0, unsharp_sigma=1.5):
    """stage 2에서 추가되는 부분: CLAHE(명암 대비) + Unsharp Masking(선명화)."""
    gray     = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    clahe    = cv2.createCLAHE(clipLimit=clahe_clip, tileGridSize=(clahe_tile, clahe_tile))
    enhanced = clahe.apply(gray)
    if unsharp_strength > 0:
        blurred  = cv2.GaussianBlur(enhanced, (0, 0), unsharp_sigma)
        enhanced = cv2.addWeighted(enhanced, 1 + unsharp_strength, blurred, -unsharp_strength, 0)
    return cv2.cvtColor(enhanced, cv2.COLOR_GRAY2BGR)


def rotate_image(img, angle):
    if angle == 0: return img
    h, w = img.shape[:2]
    cx, cy = w/2, h/2
    M = cv2.getRotationMatrix2D((cx, cy), -angle, 1.0)
    cos, sin = abs(M[0,0]), abs(M[0,1])
    new_w = int(h*sin + w*cos)
    new_h = int(h*cos + w*sin)
    M[0,2] += (new_w/2) - cx
    M[1,2] += (new_h/2) - cy
    return cv2.warpAffine(img, M, (new_w, new_h),
                          flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)


def align_to_long_axis(img):
    """세로로 긴 이미지 -> 90도 회전해 가로로 맞춤 (실제 이미지 크기 기준)."""
    h, w = img.shape[:2]
    if h > w * 1.2:
        return rotate_image(img, 90)
    return img


def upscale_if_small(img, area_thresh=71818, scale=2.0):
    """작은 이미지 -> 2배 확대 (Cubic 보간, 실제 이미지 크기 기준)."""
    h, w = img.shape[:2]
    if w * h <= area_thresh:
        img = cv2.resize(img, (int(w*scale), int(h*scale)), interpolation=cv2.INTER_CUBIC)
    return img


def prepare_stage1(img_path):
    """아무것도 없을 때: 포맷 통일만."""
    return load_raw(img_path)


def prepare_stage2(img_path):
    """+ CLAHE+unsharp만 추가 (정렬/업스케일 없음)."""
    img = load_raw(img_path)
    img = apply_clahe_unsharp(img)
    return img


def prepare_stage3(img_path):
    """+ 방향 정렬(align_to_long_axis) + upscale까지 추가.
    실제 파이프라인의 load_and_prepare와 동일한 상태 -- 다음 stage(순수 multi-angle, 이미 계산됨)
    바로 직전 단계와 맞추기 위해 align_to_long_axis도 여기 포함시켰다."""
    img = load_raw(img_path)
    img = apply_clahe_unsharp(img)
    img = align_to_long_axis(img)
    img = upscale_if_small(img)
    return img


STAGES = {
    'stage1_nothing':      prepare_stage1,
    'stage2_clahe_unsharp': prepare_stage2,
    'stage3_upscale_align': prepare_stage3,
}

## 3. 정규화 / 점수 함수 (Fix 10 적용, 관대 매칭 없음)

In [ ]:
IGNORE_IMPRINT_TOKENS = {'', 'NAN', 'NONE', 'NULL', '마크', '분할선', '없음', '무', '-', '십자'}
_RE_STRIP_TOKENS = re.compile(r'\s+|분할선|마크|\|')
_RE_ALLOWED      = re.compile(r'[^0-9A-Z가-힣+\-/]')


def normalize_imprint(text):
    if pd.isna(text):
        return ''
    text = str(text).strip().upper()
    if text in IGNORE_IMPRINT_TOKENS:
        return ''
    text = _RE_STRIP_TOKENS.sub('', text)
    text = _RE_ALLOWED.sub('', text)
    return '' if text in IGNORE_IMPRINT_TOKENS else text


def levenshtein(pred, target):
    p, t = list(pred), list(target)
    dp = list(range(len(t) + 1))
    for pc in p:
        ndp = [dp[0] + 1]
        for j, tc in enumerate(t):
            ndp.append(min(dp[j] + (pc != tc), dp[j + 1] + 1, ndp[-1] + 1))
        dp = ndp
    return dp[len(t)]


def score_one(pred, candidates):
    pred  = normalize_imprint(pred)
    valid = [c for c in candidates if c]
    if not valid:
        return None
    best_cer = min(levenshtein(pred, c) / max(len(c), 1) for c in valid)
    return max(0.0, 1.0 - best_cer)


def exact_match(pred, candidates):
    pred = normalize_imprint(pred)
    return any(pred == c for c in candidates if c)


def _extract_ocr(page):
    texts = [str(t) for t in page.get('rec_texts', [])]
    confs = [float(c) for c in page.get('rec_scores', [])]
    polys = page.get('rec_polys', [])
    if not texts:
        return '', '', float('nan'), [], []
    if polys and len(polys) == len(texts):
        def _cy(poly): return sum(p[1] for p in poly) / len(poly)
        def _cx(poly): return sum(p[0] for p in poly) / len(poly)
        heights = [max(p[1] for p in poly) - min(p[1] for p in poly) for poly in polys]
        row_thresh = (sum(heights) / len(heights)) / 2 if heights else 1
        items = sorted(zip(texts, polys, confs),
                       key=lambda x: (round(_cy(x[1]) / row_thresh), _cx(x[1])))
        texts = [t for t, _, _ in items]
        polys = [p for _, p, _ in items]
        confs = [c for _, _, c in items]
    return (' | '.join(texts), normalize_imprint(''.join(texts).strip()),
            float(np.mean(confs)) if confs else float('nan'), polys, confs)


def _build_candidates(row):
    front = normalize_imprint(row.get('print_front', ''))
    back  = normalize_imprint(row.get('print_back', ''))
    return list(dict.fromkeys(t for t in [front, back] if t))

_manifest_test['target_candidates'] = _manifest_test.apply(_build_candidates, axis=1)
candidates_map = _manifest_test.set_index('object_id')['target_candidates'].to_dict()
print(f'정답 후보(print_front/back) object_id {len(candidates_map)}개')

정답 후보(print_front/back) object_id 5088개


## 4. Pretrained PaddleOCR 초기화 (fine-tuned 미사용)

In [ ]:
from paddleocr import PaddleOCR

ocr = PaddleOCR(
    device='gpu',
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=True,
    text_detection_model_name='PP-OCRv5_server_det',
    text_recognition_model_name='PP-OCRv5_server_rec',  # pretrained (fine-tune 안 된 모델)
)
print('pretrained OCR 초기화 완료')

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
Using official model (PP-LCNet_x1_0_textline_ori), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-LCNet_x1_0_textline_ori`.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:715: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv5_server_det', None, None)
Using official model (PP-OCRv5_server_det), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-OCRv5_server_det`.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('PP-OCRv5_server_rec', None, None)
Using official model (PP-OCRv5_server_rec), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-OCRv5_server_rec`.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

pretrained OCR 초기화 완료


## 5. 전체 추론 실행 (3 stage, 전체 테스트셋)

In [ ]:
records = []
for img_path in tqdm(crop_paths, desc='OCR 실행 (stage1/2/3, 회전 탐색 없음)'):
    oid = img_path.stem
    gt_candidates = candidates_map.get(oid, [])
    result = {'object_id': oid}
    for stage_name, prep_fn in STAGES.items():
        try:
            img = prep_fn(img_path)
            ocr_result = ocr.predict(img, text_det_thresh=0.3)
            if not ocr_result:
                result[f'{stage_name}_text'] = ''
                result[f'{stage_name}_score'] = float('nan')
                result[f'{stage_name}_em'] = False
                continue
            _, norm_text, conf, _, _ = _extract_ocr(ocr_result[0])
            result[f'{stage_name}_text'] = norm_text
            result[f'{stage_name}_conf'] = conf
            if gt_candidates:
                s = score_one(norm_text, gt_candidates)
                result[f'{stage_name}_score'] = s if s is not None else float('nan')
                result[f'{stage_name}_em'] = exact_match(norm_text, gt_candidates)
            else:
                result[f'{stage_name}_score'] = float('nan')
                result[f'{stage_name}_em'] = False
        except Exception as exc:
            result[f'{stage_name}_text'] = ''
            result[f'{stage_name}_score'] = float('nan')
            result[f'{stage_name}_em'] = False
            result[f'{stage_name}_error'] = str(exc)
    records.append(result)

df_result = pd.DataFrame(records)
print(f'완료: {len(df_result)}건')

OCR 실행 (stage1/2/3, 회전 탐색 없음):   0%|          | 0/5088 [00:00<?, ?it/s]

완료: 5088건


## 6. 결과 요약

In [ ]:
print('=' * 60)
for stage_name, label in [
    ('stage1_nothing', '아무것도 없을 때'),
    ('stage2_clahe_unsharp', '+ CLAHE+unsharp'),
    ('stage3_upscale_align', '+ upscale(및 방향 정렬)'),
]:
    valid = df_result[f'{stage_name}_score'].notna()
    acc = df_result.loc[valid, f'{stage_name}_score'].mean()
    em = df_result[f'{stage_name}_em'].mean()
    print(f'{label:20s}  글자 정확도: {acc:.3f}   EM: {em:.3f}   (유효 {valid.sum()}/{len(df_result)}건)')
print('=' * 60)
print('참고 (이미 계산됨, 이 노트북에서 재계산 안 함):')
print('  순수 multi-angle             글자 정확도: 0.787   EM: 0.617')
print('  polygon 기반 회전 보정+flip   글자 정확도: 0.834   EM: 0.720')
print('=' * 60)

out_csv = RESULT_DIR / 'testset_preprocessing_stage_ablation.csv'
df_result.to_csv(out_csv, index=False)
print(f'저장 완료: {out_csv}')

아무것도 없을 때             글자 정확도: 0.458   EM: 0.334   (유효 5088/5088건)
+ CLAHE+unsharp       글자 정확도: 0.558   EM: 0.422   (유효 5088/5088건)
+ upscale(및 방향 정렬)    글자 정확도: 0.601   EM: 0.499   (유효 5088/5088건)
참고 (이미 계산됨, 이 노트북에서 재계산 안 함):
  순수 multi-angle             글자 정확도: 0.787   EM: 0.617
  polygon 기반 회전 보정+flip   글자 정확도: 0.834   EM: 0.720
저장 완료: /content/drive/MyDrive/Pillot/yoonsoo/results/testset_preprocessing_stage_ablation.csv
